In [1]:
# Cell 1 — Imports & config



import sys, os, json, struct, warnings
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm.auto import tqdm

sys.path.insert(0, str(Path("../src").resolve()))       # "." if pdb_io.py sits beside the notebook
import importlib, pdb_io; importlib.reload(pdb_io)
print("pdb_io  ->", Path(pdb_io.__file__).resolve())

ROOT = Path("../data_paderborn")
EDA  = ROOT / "eda_out"                                  # produced by notebook 01
OUT  = ROOT / "torch_out"; OUT.mkdir(parents=True, exist_ok=True)
assert EDA.exists(), f"run notebook 01 first, {EDA} missing"

CH_MAIN = ["vibration_1", "phase_current_1", "phase_current_2"]   # 64 kHz -> tensor
CH_SLOW = ["force", "speed", "torque"]                            #  4 kHz -> scalars
CH_AUX  = ["temp_2_bearing_module"]                               #  slow   -> scalar

FS_MAIN  = 64_000
WIN, HOP = 4096, 4096          # 64 ms, non-overlapping
DTYPE    = np.float32
SEED     = 0
LABELS4  = {"healthy": 0, "OR": 1, "IR": 2, "IR+OR": 3}
NAMES4   = {v: k for k, v in LABELS4.items()}
rng      = np.random.default_rng(SEED)

pdb_io  -> E:\Thesis\TCD\TCD\notebooks\pdb_io.py


In [2]:
# Cell 2 — N_KEEP from the full census (not from the 256-row feature subsample)


def resolve_n_keep():
    p = EDA / "eda_summary.json"
    if p.exists():
        j = json.load(open(p))
        for k in ("n_keep", "N_KEEP"):
            if k in j:
                return int(j[k]), f"eda_summary.json[{k}]"
    p = EDA / "census.parquet"
    if p.exists():
        c = pd.read_parquet(p)
        cand = [x for x in c.columns
                if "vibration_1" in x and any(t in x.lower() for t in ("n_sample", "len", "size"))]
        if cand:
            return int(c[cand[0]].min()), f"census.parquet[{cand[0]}] over {len(c)} files"
    f = pd.read_parquet(EDA / "features.parquet")
    cand = [x for x in f.columns if x.startswith("n_samples__") and "vibration_1" in x]
    assert cand, "no length column found anywhere -> re-run notebook 01 census cell"
    warnings.warn(f"FALLBACK: features.parquet has only {len(f)} rows (subsample) — "
                  "N_KEEP may exceed the true minimum and the export will skip short files")
    return int(f[cand[0]].min()), f"features.parquet[{cand[0]}] (SUBSAMPLE)"

N_KEEP, N_KEEP_SRC = resolve_n_keep()
N_WIN = 1 + (N_KEEP - WIN) // HOP
TAIL  = N_KEEP - ((N_WIN - 1) * HOP + WIN)      # discarded samples/file, no divisibility needed
assert N_WIN >= 1, f"WIN={WIN} > N_KEEP={N_KEEP}"
print(f"N_KEEP = {N_KEEP:,}  (source: {N_KEEP_SRC})")
print(f"windows/file = {N_WIN}   tail dropped = {TAIL} samples ({TAIL/FS_MAIN*1e3:.1f} ms)")

N_KEEP = 255,997  (source: eda_summary.json[n_keep])
windows/file = 62   tail dropped = 2045 samples (32.0 ms)


In [3]:
# Cell 3a — discover what notebook 01 actually produced (run once)



print("EDA:", EDA.resolve())
for p in sorted(EDA.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(EDA)}   {p.stat().st_size/1e6:.2f} MB")

_f = sorted(EDA.glob("feature*.parquet"))
print("\nfeature file(s):", [p.name for p in _f])
_probe = pd.read_parquet(_f[0]); print(_probe.shape); print(list(_probe.columns)[:40])
print("\n.mat files under ROOT:", sum(1 for _ in ROOT.rglob("*.mat")))

EDA: E:\Thesis\TCD\TCD\data_paderborn\eda_out
  09b_features_by_condition.png   0.05 MB
  census.parquet   0.01 MB
  eda_summary.json   0.00 MB
  features.parquet   0.17 MB
  figs\conditions.png   0.06 MB
  figs\corr.png   0.14 MB
  figs\envelope_spectra.png   0.37 MB
  figs\feature_dists.png   0.10 MB
  figs\feature_hists.png   0.08 MB
  figs\pca_tsne.png   0.17 MB
  figs\psd.png   0.35 MB
  figs\rms_by_condition.png   0.03 MB
  figs\spectrograms.png   1.50 MB
  figs\waveforms.png   0.48 MB
  file_inventory.csv   0.33 MB
  inventory.parquet   0.02 MB
  read_errors.csv   0.12 MB
  splits.json   0.00 MB
  waveforms.png   0.41 MB

feature file(s): ['features.parquet']
(256, 72)
['path', 'file_id', 'folder', 'bearing', 'trial', 'cond', 'rpm', 'torque', 'radial_force', 'component', 'origin', 'mode', 'severity', 'label4', 'vibration_1__mean', 'vibration_1__std', 'vibration_1__rms', 'vibration_1__peak', 'vibration_1__p2p', 'vibration_1__crest', 'vibration_1__shape', 'vibration_1__impulse', '

In [4]:
# Cell 3b — probe the four artifacts (run once)



import json
inv0 = pd.read_csv(EDA / "file_inventory.csv")
print("file_inventory:", inv0.shape); print(list(inv0.columns)); display(inv0.head(3))

cen = pd.read_parquet(EDA / "census.parquet")
print("\ncensus:", cen.shape); print(list(cen.columns)); display(cen.head(3))

print("\neda_summary keys:", list(json.loads((EDA/'eda_summary.json').read_text())))
print("splits keys:", list(json.loads((EDA/'splits.json').read_text())))

err = pd.read_csv(EDA / "read_errors.csv")
print("\nread_errors:", err.shape); print(list(err.columns)); display(err.head(5))

file_inventory: (2560, 14)
['path', 'file_id', 'folder', 'bearing', 'trial', 'cond', 'rpm', 'torque', 'radial_force', 'component', 'origin', 'mode', 'severity', 'label4']


,path,file_id,folder,bearing,trial,cond,rpm,torque,radial_force,component,origin,mode,severity,label4
0,..\data_paderborn\K001\N09_M07_F10_K001_1.mat,N09_M07_F10_K001_1,K001,K001,1,N09_M07_F10,900,0.7,1000,healthy,none,-,0,healthy
1,..\data_paderborn\K001\N09_M07_F10_K001_2.mat,N09_M07_F10_K001_2,K001,K001,2,N09_M07_F10,900,0.7,1000,healthy,none,-,0,healthy
2,..\data_paderborn\K001\N09_M07_F10_K001_3.mat,N09_M07_F10_K001_3,K001,K001,3,N09_M07_F10,900,0.7,1000,healthy,none,-,0,healthy



census: (96, 10)
['path', 'channels', 'error', 'n__force', 'n__phase_current_1', 'n__phase_current_2', 'n__speed', 'n__temp_2_bearing_module', 'n__torque', 'n__vibration_1']


,path,channels,error,n__force,n__phase_current_1,n__phase_current_2,n__speed,n__temp_2_bearing_module,n__torque,n__vibration_1
0,..\data_paderborn\K001\N15_M01_F10_K001_11.mat,force|phase_current_1|phase_current_2|speed|te...,,16001,256001,256001,16001,5,16001,256001
1,..\data_paderborn\K001\N09_M07_F10_K001_1.mat,force|phase_current_1|phase_current_2|speed|te...,,16008,256823,256823,16008,5,16008,256823
2,..\data_paderborn\K001\N15_M01_F10_K001_3.mat,force|phase_current_1|phase_current_2|speed|te...,,16001,256001,256001,16001,5,16001,256001



eda_summary keys: ['n_keep', 'fast_channels', 'slow_channels', 'aux_channels', 'fs_fast', 'fs_slow', 'n_files_total', 'n_files_featured', 'feature_cols', 'measured_cols', 'label_col', 'group_col', 'seed']
splits keys: ['artificial_to_real', 'real_bearing_holdout', 'cross_condition']

read_errors: (2560, 2)
['path', 'err']


,path,err
0,..\data_paderborn\K001\N09_M07_F10_K001_1.mat,NaN
1,..\data_paderborn\K001\N09_M07_F10_K001_2.mat,NaN
2,..\data_paderborn\K001\N09_M07_F10_K001_3.mat,NaN
3,..\data_paderborn\K001\N09_M07_F10_K001_4.mat,NaN
4,..\data_paderborn\K001\N09_M07_F10_K001_5.mat,NaN


In [5]:
# Cell 3 — Inventory, column resolution, label4



import re

FEAT_PATH = sorted(EDA.glob("feature*.parquet"))
assert FEAT_PATH, f"no features parquet in {EDA}"
feat = pd.read_parquet(FEAT_PATH[0])
print(f"features: {FEAT_PATH[0].name} {feat.shape}")

# ---------- 1. try existing artifacts (must be the FULL list, not the 256-row subsample) ----
INV_NAMES = ["inventory.parquet", "files.parquet", "file_index.parquet", "manifest.parquet",
             "index.parquet", "census.parquet", "inventory.csv", "files.csv"]

def load_inventory():
    for n in INV_NAMES:
        p = EDA / n
        if not p.exists(): continue
        df = pd.read_csv(p) if p.suffix == ".csv" else pd.read_parquet(p)
        if len(df) > 300:
            print(f"inventory <- {p.name} {df.shape}"); return df
        print(f"skip {p.name}: only {len(df)} rows (subsample)")
    return None

# ---------- 2. rebuild from filenames -------------------------------------------------------
REAL = {"KA04","KA15","KA16","KA22","KA30","KI04","KI14","KI16","KI17","KI18","KI21",
        "KB23","KB24","KB27"}                                  # accelerated lifetime tests
ARTI = {"KA01","KA03","KA05","KA06","KA07","KA08","KA09",
        "KI01","KI03","KI05","KI07","KI08"}                    # EDM / drilling / engraving

def comp_of(code):
    c = code.upper()
    if c.startswith("K0"): return "healthy"
    if c.startswith("KA"): return "OR"
    if c.startswith("KI"): return "IR"
    if c.startswith("KB"): return "IR+OR"
    raise ValueError(f"unknown bearing code {code!r}")

def origin_of(code):
    c = code.upper()
    return "healthy" if c.startswith("K0") else ("real" if c in REAL else
           ("artificial" if c in ARTI else "unknown"))

def scan_raw():
    skip, rows = {EDA.resolve(), OUT.resolve()}, []
    for p in ROOT.rglob("*.mat"):
        rp = p.resolve()
        if any(s in rp.parents for s in skip): continue
        q = p.stem.split("_")
        if len(q) < 5: continue
        n, m, f, code, tr = q[0].upper(), q[1].upper(), q[2].upper(), q[-2].upper(), q[-1]
        if not (n.startswith("N") and m.startswith("M") and f.startswith("F")
                and code.startswith("K")): continue
        rows.append(dict(path=p.relative_to(ROOT).as_posix(), bearing=code,
                         cond=f"{n}_{m}_{f}", trial=int(re.sub(r"\D", "", tr) or 0),
                         component=comp_of(code), origin=origin_of(code),
                         speed_set=int(n[1:]) * 100,      # N09 -> 900 rpm
                         torque_set=int(m[1:]) / 10,      # M07 -> 0.7 Nm
                         force_set=int(f[1:]) * 100))     # F10 -> 1000 N
    return pd.DataFrame(rows)

files = load_inventory()
if files is None:
    files = scan_raw()
    assert len(files), f"no Paderborn .mat found under {ROOT.resolve()} — check ROOT"
    files = files.sort_values(["bearing", "cond", "trial"]).reset_index(drop=True)
    files.to_parquet(EDA / "inventory.parquet", index=False)
    print(f"REBUILT inventory from filesystem -> {EDA/'inventory.parquet'} {files.shape}")

# ---------- 3. resolve columns (unchanged logic) --------------------------------------------
def pick(df, *cands, required=True):
    for c in cands:
        if c in df.columns: return c
    if required: raise KeyError(f"none of {cands} in inventory: {list(df.columns)}")
    return None

COL = {k: pick(files, *v, required=req) for k, v, req in [
    ("path",      ("path", "filepath", "file", "fullpath", "rel_path"), True),
    ("bearing",   ("bearing", "bearing_code", "code", "bearing_id"),    True),
    ("cond",      ("cond", "condition", "setting", "op_cond"),          True),
    ("trial",     ("trial", "rep", "measurement", "run"),               False),
    ("component", ("component", "damage_component", "class", "fault"),  False),
    ("origin",    ("origin", "damage_origin", "kind", "source"),        False),
    ("label",     ("label4", "label", "y"),                             False),
    ("torque_set",("torque_set", "torque", "torque_nm_nominal"),        False)]}
print(COL)

def norm_comp(s):
    t = str(s).strip().lower().replace(" ", "").replace("_", "")
    if t in {"healthy","k0","none","ok","0"}:                   return "healthy"
    if t in {"ir+or","or+ir","combined","both","irandor","3"}:  return "IR+OR"
    if t.startswith("ir") or "inner" in t or t == "2":          return "IR"
    if t.startswith("or") or "outer" in t or t == "1":          return "OR"
    raise ValueError(f"unmapped component: {s!r}")

if COL["label"] and pd.api.types.is_integer_dtype(files[COL["label"]]):
    files["label"] = files[COL["label"]].astype(int)
    files["label_name"] = files["label"].map(NAMES4)
else:
    src = COL["label"] or COL["component"]
    assert src, "need a component/label column"
    files["label_name"] = files[src].map(norm_comp)
    files["label"] = files["label_name"].map(LABELS4).astype(int)

files["path_abs"] = files[COL["path"]].map(
    lambda p: str(Path(p) if Path(p).is_absolute() else (ROOT / p)))
files["bearing"] = files[COL["bearing"]].astype(str)
files["cond"]    = files[COL["cond"]].astype(str)
files["trial"]   = files[COL["trial"]] if COL["trial"] else 0
files["origin"]  = files[COL["origin"]].astype(str) if COL["origin"] else "unknown"

miss = [p for p in files.path_abs if not Path(p).exists()]
assert not miss, f"{len(miss)} paths do not exist, e.g. {miss[:3]}"
print(f"OK: {len(files)} files, {files.bearing.nunique()} bearings, "
      f"{files.cond.nunique()} conditions, unknown-origin={int((files.origin=='unknown').sum())}")

display(files.pivot_table(index="label_name", columns="cond", aggfunc="size", fill_value=0))
display(files.groupby(["origin", "label_name"]).size().unstack(fill_value=0))

features: features.parquet (256, 72)
inventory <- inventory.parquet (2560, 9)
{'path': 'path', 'bearing': 'bearing', 'cond': 'cond', 'trial': 'trial', 'component': 'component', 'origin': 'origin', 'label': None, 'torque_set': 'torque_set'}
OK: 2560 files, 32 bearings, 4 conditions, unknown-origin=0


cond,N09_M07_F10,N15_M01_F10,N15_M07_F04,N15_M07_F10
label_name,,,,
IR,220,220,220,220
IR+OR,60,60,60,60
OR,240,240,240,240
healthy,120,120,120,120


label_name,IR,IR+OR,OR,healthy
origin,,,,
artificial,400,0,560,0
healthy,0,0,0,480
real,480,240,400,0


In [6]:
# Cell 4 — Three split strategies, all bearing-grouped, stored as columns



def bearing_holdout(df, seed=SEED):
    r, m = np.random.default_rng(seed), {}
    for _, g in df.groupby("label_name"):
        b = r.permutation(g.bearing.unique())
        i, j = max(1, int(.6*len(b))), max(2, int(.8*len(b)))
        m |= {x: "train" for x in b[:i]} | {x: "val" for x in b[i:j]} | {x: "test" for x in b[j:]}
    return df.bearing.map(m)

def artificial_to_real(df, seed=SEED):
    o = df.origin.str.lower()
    if not (o.str.contains("real").any() and o.str.contains("artif").any()):
        warnings.warn("origin lacks real/artificial -> split_art2real disabled")
        return pd.Series("unused", index=df.index)
    r = np.random.default_rng(seed)
    heal = df.loc[df.label_name == "healthy", "bearing"].unique()
    hv = set(r.permutation(heal)[:max(1, len(heal)//3)])
    def f(row):
        if "real"    in row.origin.lower(): return "test"
        if row.label_name == "healthy":     return "val" if row.bearing in hv else "train"
        return "train"
    return df.apply(f, axis=1)

def cross_condition(df, holdout=None, seed=SEED):
    conds = sorted(df.cond.unique())
    holdout = holdout or conds[-1:]
    r = np.random.default_rng(seed)
    tr = df[~df.cond.isin(holdout)]
    vb = set(r.permutation(tr.bearing.unique())[:max(1, tr.bearing.nunique()//5)])
    return df.apply(lambda x: "test" if x.cond in holdout
                    else ("val" if x.bearing in vb else "train"), axis=1)

files["split_bearing"]   = bearing_holdout(files)
files["split_art2real"]  = artificial_to_real(files)
files["split_crosscond"] = cross_condition(files)
SPLIT_COLS = [c for c in ("split_bearing", "split_art2real", "split_crosscond")
              if files[c].nunique() > 1 and (files[c] != "unused").all()]

for c in SPLIT_COLS:
    print(f"\n== {c} ==")
    display(files.pivot_table(index=c, columns="label_name", aggfunc="size", fill_value=0))


== split_bearing ==


label_name,IR,IR+OR,OR,healthy
split_bearing,,,,
test,240,80,240,160
train,480,80,560,240
val,160,80,160,80



== split_art2real ==


label_name,IR,IR+OR,OR,healthy
split_art2real,,,,
test,480,240,400,0
train,400,0,560,320
val,0,0,0,160



== split_crosscond ==


label_name,IR,IR+OR,OR,healthy
split_crosscond,,,,
test,220,60,240,120
train,540,180,600,240
val,120,0,120,120


In [7]:
# Cell 5 — pdb_io adapter + time-correct slow-channel window means



# --- ADJUST THIS ONE FUNCTION if pdb_io.read_pdb_mat has a different signature -------------
def load_channels(path, channels):
    """-> sig {name: 1-D float32}, t {name: 1-D float64 | None}.
       NOTE: never pass a single n_keep for mixed sample rates — it would truncate the
       4 kHz channels to the 64 kHz length. Truncation happens per rate, below."""
    try:
        out = pdb_io.read_pdb_mat(path, channels=channels, with_time=True)
    except TypeError:
        out = pdb_io.read_pdb_mat(path, channels=channels)
    t = {}
    if isinstance(out, tuple) and len(out) == 2:
        out, t = out
    sig = {}
    for k, v in out.items():
        if isinstance(v, dict):                       # {"x": ..., "t": ...}
            sig[k] = np.asarray(v.get("x", v.get("data")), np.float32).ravel()
            if v.get("t") is not None: t[k] = np.asarray(v["t"], np.float64).ravel()
        else:
            sig[k] = np.asarray(v, np.float32).ravel()
    return sig, {k: np.asarray(v, np.float64).ravel() for k, v in t.items() if v is not None}

ALL_CH = CH_MAIN + CH_SLOW + CH_AUX
_probe_sig, _probe_t = load_channels(files.path_abs.iloc[0], ALL_CH)
print({k: v.shape for k, v in _probe_sig.items()})
print("time vectors:", {k: v.shape for k, v in _probe_t.items()} or "none -> proportional mapping")
missing = [c for c in ALL_CH if c not in _probe_sig]
assert not missing, f"channels missing from reader output: {missing}"

def time_of(name, sig, t):
    """Time vector for a channel: from root.X if pdb_io returned it, else assume the
       channel spans the same wall-clock duration as CH_MAIN[0] (true for Paderborn)."""
    if name in t and len(t[name]) == len(sig[name]):
        return t[name]
    n_ref = len(sig[CH_MAIN[0]]); n = len(sig[name])
    return np.arange(n, dtype=np.float64) * (n_ref / max(n, 1)) / FS_MAIN

def window_means(x, t_slow, t_fast, n_win, win, hop):
    """Mean of a slow channel over each fast-channel window's time span (vectorized)."""
    s  = np.arange(n_win) * hop
    lo, hi = t_fast[s], t_fast[s + win - 1]
    i0 = np.searchsorted(t_slow, lo, "left")
    i1 = np.clip(np.searchsorted(t_slow, hi, "right"), i0 + 1, len(x))
    cs = np.concatenate([[0.0], np.cumsum(x, dtype=np.float64)])
    return (cs[i1] - cs[i0]) / (i1 - i0)

def windows_of(x, n_win, win, hop):
    if hop == win:                                   # cheap path, no stride tricks
        return x[:n_win * win].reshape(n_win, win)
    return np.lib.stride_tricks.sliding_window_view(x, win)[::hop][:n_win]

{'force': (16008,), 'phase_current_1': (256823,), 'phase_current_2': (256823,), 'speed': (16008,), 'temp_2_bearing_module': (5,), 'torque': (16008,), 'vibration_1': (256823,)}
time vectors: none -> proportional mapping


In [8]:
# Cell 6 — Export with skip-and-truncate (no silent zero blocks)



def shrink_npy(path, n_rows):
    """Rewrite the .npy header for a smaller leading dim and truncate the file in place."""
    with open(path, "r+b") as fh:
        ver = np.lib.format.read_magic(fh)
        shape, fortran, dt = (np.lib.format.read_array_header_1_0(fh) if ver == (1, 0)
                              else np.lib.format.read_array_header_2_0(fh))
        data_start = fh.tell()
        if n_rows == shape[0]: return shape
        new = (int(n_rows),) + tuple(shape[1:])
        d = {"descr": np.lib.format.dtype_to_descr(dt), "fortran_order": bool(fortran), "shape": new}
        s, hlen = str(d), 8 + (2 if ver == (1, 0) else 4)
        avail = data_start - hlen
        assert len(s) + 1 <= avail, "header grew; cannot shrink in place"
        s = s + " " * (avail - len(s) - 1) + "\n"
        fh.seek(hlen - (2 if ver == (1, 0) else 4))
        fh.write(struct.pack("<H" if ver == (1, 0) else "<I", len(s)))
        fh.write(s.encode("latin1"))
    os.truncate(path, data_start + n_rows * int(np.prod(new[1:])) * np.dtype(dt).itemsize)
    return new

ARR = OUT / "windows.npy"
N_ALLOC, C = len(files) * N_WIN, len(CH_MAIN)
print(f"allocating {N_ALLOC:,} x {C} x {WIN} float32 = "
      f"{N_ALLOC*C*WIN*4/1e9:.2f} GB (trimmed after export)")

X = np.lib.format.open_memmap(ARR, mode="w+", dtype=DTYPE, shape=(N_ALLOC, C, WIN))
meta_rows, skipped, cur = [], [], 0

ALLOW_SHORT = True          # short file -> fewer windows, instead of being dropped

n_exported_files = 0

for fid, r in enumerate(tqdm(files.itertuples(), total=len(files), desc="export")):

    try:
        sig, tv = load_channels(r.path_abs, ALL_CH)
        n_av = min(len(sig[c]) for c in CH_MAIN)
        if n_av < WIN: raise ValueError(f"only {n_av} samples < WIN")
        if n_av < N_KEEP and not ALLOW_SHORT: raise ValueError(f"{n_av} < N_KEEP")
        n_f   = min(n_av, N_KEEP)
        n_w   = 1 + (n_f - WIN) // HOP
        main  = np.stack([sig[c][:n_f] for c in CH_MAIN])
        if not np.isfinite(main).all(): raise ValueError("non-finite samples in CH_MAIN")
        blk = np.stack([windows_of(main[i], n_w, WIN, HOP) for i in range(len(CH_MAIN))], axis=1)
    except Exception as e:
        skipped.append({"file_id": fid, "path": r.path_abs, "reason": repr(e)}); continue

    X[cur:cur + n_w] = blk.astype(DTYPE, copy=False)
    t_fast = time_of(CH_MAIN[0], sig, tv)[:n_f]
    ops = {c: window_means(sig[c], time_of(c, sig, tv), t_fast, n_w, WIN, HOP) for c in CH_SLOW}
    temp = float(np.mean(sig[CH_AUX[0]])) if len(sig[CH_AUX[0]]) else np.nan

    for w in range(n_w):
        meta_rows.append(dict(
            idx=cur + w, file_id=fid, window=w, start=w * HOP, n_win_file=n_w,
            bearing=r.bearing, cond=r.cond, trial=r.trial, origin=r.origin,
            label=int(r.label), label_name=r.label_name,
            speed_rpm=float(ops["speed"][w]), torque_nm=float(ops["torque"][w]),
            force_n=float(ops["force"][w]), temp_c=temp,
            torque_set=(getattr(r, COL["torque_set"]) if COL["torque_set"] else np.nan),
            **{c: getattr(r, c) for c in SPLIT_COLS}))
    cur += n_w
    n_exported_files += 1

X.flush(); del X
shape = shrink_npy(ARR, cur)
meta = pd.DataFrame(meta_rows)
meta.to_parquet(OUT / "meta.parquet", index=False)
pd.DataFrame(skipped).to_csv(OUT / "skipped_files.csv", index=False)

# print(f"wrote {cur:,} windows from {cur//N_WIN}/{len(files)} files -> {shape}")
print(
    f"wrote {cur:,} windows from "
    f"{n_exported_files}/{len(files)} files -> {shape}"
)
print(f"skipped {len(skipped)} files"); display(pd.DataFrame(skipped).head())

allocating 158,720 x 3 x 4096 float32 = 7.80 GB (trimmed after export)


export:   0%|          | 0/2560 [00:00<?, ?it/s]

wrote 158,656 windows from 2559/2560 files -> (158656, 3, 4096)
skipped 1 files


,file_id,path,reason
0,981,..\data_paderborn\KA08\N15_M01_F10_KA08_2.mat,TypeError('Expecting matrix here')


In [9]:
# Cell 7 — Train-only normalization stats + dataset_config.json


Xr  = np.load(ARR, mmap_mode="r")
meta = pd.read_parquet(OUT / "meta.parquet")

norm = {}
for s in SPLIT_COLS:
    ids = meta.loc[meta[s] == "train", "idx"].to_numpy()
    sub = np.sort(np.random.default_rng(SEED).permutation(ids)[:20_000])
    a = np.asarray(Xr[sub], dtype=np.float64)
    norm[s] = {"mean": a.mean((0, 2)).tolist(), "std": (a.std((0, 2)) + 1e-8).tolist(),
               "n_used": int(len(sub))}

cfg = dict(array="windows.npy", meta="meta.parquet", shape=list(Xr.shape), dtype="float32",
           channels=CH_MAIN, slow_channels=CH_SLOW, aux_channels=CH_AUX,
           cond_cols=["speed_rpm", "torque_nm", "force_n", "temp_c"],
           fs=FS_MAIN, win=WIN, hop=HOP, n_keep=N_KEEP, n_keep_source=N_KEEP_SRC,
           n_windows_per_file=N_WIN, tail_dropped=int(TAIL),
           class_names={str(k): v for k, v in NAMES4.items()},
           splits=SPLIT_COLS, norm=norm, seed=SEED,
           n_files_exported=int(cur // N_WIN), n_files_skipped=len(skipped))
json.dump(cfg, open(OUT / "dataset_config.json", "w"), indent=2)
# display(pd.DataFrame(norm[SPLIT_COLS[0]], index=CH_MAIN + ["-"])[["mean", "std"]].head(len(CH_MAIN)))

print(f"\nNormalization: {SPLIT_COLS[0]}")
for ch, mu, sd in zip(
    CH_MAIN,
    norm[SPLIT_COLS[0]]["mean"],
    norm[SPLIT_COLS[0]]["std"]
):
    print(f"{ch:25s} mean={mu: .6f}  std={sd: .6f}")




Normalization: split_bearing
vibration_1               mean= 0.002755  std= 0.327618
phase_current_1           mean=-0.017899  std= 1.563402
phase_current_2           mean= 0.023186  std= 1.575398


In [12]:
# Cell 8 — Dataset (lazy memmap, fork-safe) + DataLoader with balanced sampler



class PaderbornWindows(Dataset):
    """Memmapped windows -> (C, WIN). target='label4'|'binary'. norm='global'|'instance'|None."""
    def __init__(self, root=OUT, split="train", strategy="split_bearing", norm="global",
                 target="label4", return_cond=False, train=False, jitter=0, noise=0.0):
        self.root = Path(root)
        self.cfg  = json.load(open(self.root / "dataset_config.json"))
        m = pd.read_parquet(self.root / self.cfg["meta"])
        self.m = (m if split is None else m[m[strategy] == split]).reset_index(drop=True)
        assert len(self.m), f"empty split {strategy}={split}"
        st = self.cfg["norm"][strategy]
        self.mu = torch.tensor(st["mean"], dtype=torch.float32)[:, None]
        self.sd = torch.tensor(st["std"],  dtype=torch.float32)[:, None]
        self.ids = self.m.idx.to_numpy()
        y = torch.tensor(self.m.label.to_numpy(), dtype=torch.long)
        self.y = (y > 0).long() if target == "binary" else y
        self.n_classes = 2 if target == "binary" else len(self.cfg["class_names"])
        self.cond = torch.tensor(self.m[self.cfg["cond_cols"]].to_numpy(np.float32))
        self.norm, self.return_cond = norm, return_cond
        self.train, self.jitter, self.noise = train, jitter, noise
        self._X = None                                   # opened per worker -> fork/spawn safe

    def __len__(self): return len(self.ids)

    def __getitem__(self, i):
        if self._X is None:
            self._X = np.load(self.root / self.cfg["array"], mmap_mode="r")
        x = torch.from_numpy(np.array(self._X[self.ids[i]], dtype=np.float32))
        if   self.norm == "global":   x = (x - self.mu) / self.sd
        elif self.norm == "instance": x = (x - x.mean(1, True)) / (x.std(1, True) + 1e-8)
        if self.train:
            if self.jitter: x = torch.roll(x, int(np.random.randint(-self.jitter, self.jitter + 1)), -1)
            if self.noise and np.random.rand() < .5: x = x + torch.randn_like(x) * self.noise
        return (x, self.cond[i], self.y[i]) if self.return_cond else (x, self.y[i])

'''
STRATEGY, TARGET = "split_bearing", "label4"
tr = PaderbornWindows(split="train", strategy=STRATEGY, target=TARGET,
                      train=True, jitter=256, noise=0.01)
va = PaderbornWindows(split="val",  strategy=STRATEGY, target=TARGET)
te = PaderbornWindows(split="test", strategy=STRATEGY, target=TARGET)

yt = tr.y.numpy(); w = (1.0 / np.bincount(yt, minlength=tr.n_classes).clip(1))[yt]
dl = {"train": DataLoader(tr, batch_size=128, num_workers=4, pin_memory=True,
                          persistent_workers=True, drop_last=True,
                          sampler=WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double),
                                                        len(w), replacement=True)),
      **{k: DataLoader(d, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
         for k, d in [("val", va), ("test", te)]}}

xb, yb = next(iter(dl["train"]))
print(len(tr), len(va), len(te), "|", tuple(xb.shape),
      f"mean={xb.mean():.3f} std={xb.std():.3f}", "| batch labels:", torch.bincount(yb).tolist())
'''




# ============================================================
# CELL 8 — FINAL DATASETS + SAFE DATALOADERS
# ============================================================

STRATEGY = "split_bearing"
TARGET = "label4"

tr = PaderbornWindows(
    OUT,
    split="train",
    strategy=STRATEGY,
    target=TARGET,
    return_cond=False,
    train=True,
    jitter=256,
    noise=0.01,
)

va = PaderbornWindows(
    OUT,
    split="val",
    strategy=STRATEGY,
    target=TARGET,
    return_cond=False,
    train=False,
)

te = PaderbornWindows(
    OUT,
    split="test",
    strategy=STRATEGY,
    target=TARGET,
    return_cond=False,
    train=False,
)

print("train:", len(tr))
print("val:  ", len(va))
print("test: ", len(te))

# Class-balanced sampling
yt = tr.y.numpy()

class_counts = np.bincount(yt, minlength=tr.n_classes)
print("class counts:", class_counts.tolist())

class_weights = 1.0 / np.maximum(class_counts, 1)
sample_weights = class_weights[yt]

sampler = WeightedRandomSampler(
    torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)

# IMPORTANT:
# num_workers=0 is intentional on Windows/Jupyter.
# We already verified that this is fast (~0.12 sec/batch).
dl = {
    "train": DataLoader(
        tr,
        batch_size=128,
        num_workers=0,
        pin_memory=False,
        drop_last=True,
        sampler=sampler,
    ),

    "val": DataLoader(
        va,
        batch_size=128,
        shuffle=False,
        num_workers=0,
        pin_memory=False,
    ),

    "test": DataLoader(
        te,
        batch_size=128,
        shuffle=False,
        num_workers=0,
        pin_memory=False,
    ),
}

# Test the exact loader that Cell 9 will use
t0 = time.time()
xb, yb = next(iter(dl["train"]))

print("batch:", tuple(xb.shape))
print("labels:", torch.bincount(yb, minlength=tr.n_classes).tolist())
print("time:", time.time() - t0, "sec")

train: 84256
val:   29760
test:  44640
class counts: [14880, 34656, 29760, 4960]
batch: (128, 3, 4096)
labels: [38, 36, 33, 21]
time: 0.0821387767791748 sec


In [13]:
# Cell 9 — 1D-CNN smoke test



class CNN1D(nn.Module):
    def __init__(self, in_ch=3, n_cls=4, w=32):
        super().__init__()
        def blk(i, o, k=9, s=2):
            return nn.Sequential(nn.Conv1d(i, o, k, s, k//2, bias=False),
                                 nn.BatchNorm1d(o), nn.ReLU(inplace=True))
        self.f = nn.Sequential(blk(in_ch, w, 65, 4), blk(w, 2*w), nn.MaxPool1d(2),
                               blk(2*w, 4*w), nn.MaxPool1d(2), blk(4*w, 8*w),
                               nn.AdaptiveAvgPool1d(1), nn.Flatten())
        self.c = nn.Sequential(nn.Dropout(.3), nn.Linear(8*w, n_cls))
    def forward(self, x): return self.c(self.f(x))

dev  = "cuda" if torch.cuda.is_available() else "cpu"
net  = CNN1D(len(CH_MAIN), tr.n_classes).to(dev)
opt  = torch.optim.AdamW(net.parameters(), 1e-3, weight_decay=1e-4)
crit = nn.CrossEntropyLoss(label_smoothing=0.05)

net.train()
for i, (x, y) in enumerate(dl["train"]):
    x, y = x.to(dev), y.to(dev)
    opt.zero_grad(set_to_none=True)
    out = net(x); loss = crit(out, y); loss.backward(); opt.step()
    print(f"step {i}  loss {loss.item():.4f}  acc {(out.argmax(1)==y).float().mean():.3f}")
    if i == 9: break
print("params:", sum(p.numel() for p in net.parameters())/1e6, "M | device:", dev)

step 0  loss 1.4095  acc 0.297
step 1  loss 1.2434  acc 0.523
step 2  loss 1.2165  acc 0.469
step 3  loss 1.0968  acc 0.609
step 4  loss 1.0800  acc 0.602
step 5  loss 0.9577  acc 0.625
step 6  loss 0.9243  acc 0.586
step 7  loss 0.9066  acc 0.641
step 8  loss 0.8601  acc 0.680
step 9  loss 0.7435  acc 0.719
params: 0.3953 M | device: cuda


In [14]:
# Cell 10 — Leakage & integrity verification


'''
changed it due to discrepenacy between split leackage strategies split_art2real and split_crosscond, which allow bearing overlap but not file overlap

m   = pd.read_parquet(OUT / "meta.parquet")
cfg = json.load(open(OUT / "dataset_config.json"))
Xr  = np.load(OUT / cfg["array"], mmap_mode="r")

for s in cfg["splits"]:
    bs = {k: set(g.bearing)  for k, g in m.groupby(s)}
    fs = {k: set(g.file_id)  for k, g in m.groupby(s)}
    ov_b = {f"{a}|{b}": len(bs[a] & bs[b]) for a in bs for b in bs if a < b}
    ov_f = {f"{a}|{b}": len(fs[a] & fs[b]) for a in fs for b in fs if a < b}
    print(f"{s:18s} bearing overlap {ov_b}  file overlap {ov_f}")
    assert max(ov_b.values()) == 0 and max(ov_f.values()) == 0, f"LEAKAGE in {s}"

assert m.idx.is_unique and m.idx.max() + 1 == Xr.shape[0] == len(m)
assert np.isfinite(np.asarray(Xr[np.sort(rng.choice(len(m), 2000, replace=False))])).all()
# assert (m.groupby("file_id").size() == cfg["n_windows_per_file"]).all()
#  relax the fixed-count assert
cnt = m.groupby("file_id").size()
assert cnt.between(1, cfg["n_windows_per_file"]).all()
print("windows/file:", cnt.min(), "-", cnt.max(), "| files short of N_KEEP:",
      int((cnt < cfg["n_windows_per_file"]).sum()))

nz = np.abs(np.asarray(Xr[np.sort(rng.choice(len(m), 2000, replace=False))])).sum((1, 2))
assert (nz > 0).all(), "all-zero windows -> unwritten memmap rows survived truncation"




print("\nmeasured operating point per split:")
display(m.groupby("cond")[cfg["cond_cols"]].agg(["mean", "std"]).round(2))
display(m.groupby([cfg["splits"][0], "label_name"]).size().unstack(fill_value=0))
print("OK")'''


m   = pd.read_parquet(OUT / "meta.parquet")
cfg = json.load(open(OUT / "dataset_config.json"))
Xr  = np.load(OUT / cfg["array"], mmap_mode="r")

for s in cfg["splits"]:

    groups = {
        k: g for k, g in m.groupby(s)
        if k in ("train", "val", "test")
    }

    bs = {k: set(g.bearing) for k, g in groups.items()}
    fs = {k: set(g.file_id) for k, g in groups.items()}

    ov_b = {
        f"{a}|{b}": len(bs[a] & bs[b])
        for a in bs for b in bs if a < b
    }

    ov_f = {
        f"{a}|{b}": len(fs[a] & fs[b])
        for a in fs for b in fs if a < b
    }

    print(f"\n{s}")
    print("  bearing overlap:", ov_b)
    print("  file overlap:   ", ov_f)

    # File-level overlap is NEVER allowed.
    assert max(ov_f.values(), default=0) == 0, \
        f"FILE LEAKAGE in {s}"

    # Bearing overlap depends on the experimental protocol.
    if s in ("split_bearing", "split_art2real"):
        assert max(ov_b.values(), default=0) == 0, \
            f"BEARING LEAKAGE in {s}"

    elif s == "split_crosscond":
        print("  bearing overlap allowed: cross-condition protocol")

print("\nIntegrity checks")

assert m.idx.is_unique
assert m.idx.max() + 1 == Xr.shape[0] == len(m)

sample_idx = np.sort(
    rng.choice(len(m), min(2000, len(m)), replace=False)
)

assert np.isfinite(np.asarray(Xr[sample_idx])).all()

cnt = m.groupby("file_id").size()

assert cnt.between(1, cfg["n_windows_per_file"]).all()

print(
    "windows/file:",
    cnt.min(), "-", cnt.max(),
    "| files short of N_KEEP:",
    int((cnt < cfg["n_windows_per_file"]).sum())
)

nz = np.abs(np.asarray(Xr[sample_idx])).sum((1, 2))
assert (nz > 0).all(), \
    "all-zero windows -> unwritten memmap rows survived truncation"

print("\nMeasured operating point per condition:")
display(
    m.groupby("cond")[cfg["cond_cols"]]
     .agg(["mean", "std"])
     .round(2)
)

display(
    m.groupby([cfg["splits"][0], "label_name"])
     .size()
     .unstack(fill_value=0)
)

print("\nOK")


split_bearing
  bearing overlap: {'test|train': 0, 'test|val': 0, 'train|val': 0}
  file overlap:    {'test|train': 0, 'test|val': 0, 'train|val': 0}

split_art2real
  bearing overlap: {'test|train': 0, 'test|val': 0, 'train|val': 0}
  file overlap:    {'test|train': 0, 'test|val': 0, 'train|val': 0}

split_crosscond
  bearing overlap: {'test|train': 26, 'test|val': 6, 'train|val': 0}
  file overlap:    {'test|train': 0, 'test|val': 0, 'train|val': 0}
  bearing overlap allowed: cross-condition protocol

Integrity checks
windows/file: 61 - 62 | files short of N_KEEP: 2

Measured operating point per condition:


speed_rpm       torque_nm        force_n        temp_c      
                 mean   std      mean   std     mean    std   mean   std
cond                                                                    
N09_M07_F10    899.78  6.30      1.23  0.04  1032.29  43.00  46.75  4.42
N15_M01_F10   1499.62  0.87      0.62  0.03  1048.84  40.94  45.99  4.52
N15_M07_F04   1499.59  1.11      1.25  0.02   419.17  41.40  46.87  5.00
N15_M07_F10   1499.59  0.87      1.26  0.02  1031.41  37.72  48.31  4.54

label_name,IR,IR+OR,OR,healthy
split_bearing,,,,
test,14880,4960,14880,9920
train,29760,4960,34656,14880
val,9920,4960,9920,4960



OK
